# Econometría II — Tarea 1

Notebook simple y directo para resolver las cinco preguntas con `NLS80V2.dta`.

El código funciona en Google Colab. Si la base no está disponible, aparecerá una ventana para subirla.

## Cargar la base y las librerías

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Rutas habituales: Colab, la tarea original o la carpeta del repositorio.
rutas = [
    Path('/content/NLS80V2.dta'),
    Path('/mnt/data/NLS80V2.dta'),
    Path('data/NLS80V2.dta'),
]
ruta_extra = os.environ.get('NLS80V2_PATH')
if ruta_extra:
    rutas.append(Path(ruta_extra))

ruta = next((archivo for archivo in rutas if archivo.is_file()), None)

# Si se abre directamente en Colab, permite subir la base.
if ruta is None:
    try:
        from google.colab import files
        archivo_subido = files.upload()
        ruta = Path(next(iter(archivo_subido)))
    except ImportError:
        raise FileNotFoundError('Ubica NLS80V2.dta en la carpeta data/.')

datos = pd.read_stata(ruta)
datos['exper2'] = datos['exper'] ** 2
print(f'Base cargada: {len(datos)} observaciones')

Base cargada: 935 observaciones


## 1. Regresión del logaritmo del salario

Estimamos:

$$lwage=\beta_0+\beta_1 educ+\beta_2 exper+\beta_3 exper^2+u.$$

In [2]:
X = sm.add_constant(datos[['educ', 'exper', 'exper2']])
modelo = sm.OLS(datos['lwage'], X).fit()

tabla = pd.DataFrame({
    'Coeficiente': modelo.params,
    'Error estándar': modelo.bse
})
print(tabla.round(5))

        Coeficiente  Error estándar
const       5.05292         0.21333
educ        0.08383         0.00725
exper       0.06710         0.02394
exper2     -0.00158         0.00084


**Conclusión:** la educación tiene un coeficiente de **0,08383** y un error estándar de **0,00725**. Manteniendo lo demás constante, un año adicional de educación se asocia con un salario aproximadamente 8,4 % mayor.

## 2. Aumentar un año la educación de todos

La base cumple `age = educ + exper + 6`. Por eso, a edad fija, aumentar educación en un año reduce experiencia en un año. El efecto promedio es:

$$\beta_1-\beta_2+\beta_3(1-2\overline{exper}).$$

In [3]:
exper_promedio = datos['exper'].mean()
contraste = [0, 1, -1, 1 - 2 * exper_promedio]
prueba = modelo.t_test(contraste)

efecto_log = prueba.effect.item()
ee_efecto_log = prueba.sd.item()
porcentaje = 100 * (np.exp(efecto_log) - 1)

print(f'Efecto en log salario: {efecto_log:.5f}')
print(f'Error estándar: {ee_efecto_log:.5f}')
print(f'Efecto porcentual: {porcentaje:.2f}%')

Efecto en log salario: 0.05822
Error estándar: 0.00596
Efecto porcentual: 6.00%


**Conclusión:** el log salario promedio aumenta **0,05822** puntos, equivalente aproximadamente a **6,00 %**. Su error estándar es **0,00596**.

## 3. La misma respuesta con covariables redefinidas

Creamos dos variables nuevas. Con esta parametrización, el coeficiente de `educ` es directamente el efecto de la pregunta 2.

In [4]:
a = 1 - 2 * exper_promedio
datos['educ_mas_exper'] = datos['educ'] + datos['exper']
datos['exper2_ajustada'] = datos['exper2'] - a * datos['educ']

X_nueva = sm.add_constant(datos[['educ', 'educ_mas_exper', 'exper2_ajustada']])
modelo_nuevo = sm.OLS(datos['lwage'], X_nueva).fit()

print(f'Coeficiente de educ: {modelo_nuevo.params["educ"]:.5f}')
print(f'Error estándar: {modelo_nuevo.bse["educ"]:.5f}')

Coeficiente de educ: 0.05822
Error estándar: 0.00596


**Conclusión:** obtenemos nuevamente **0,05822** con error estándar **0,00596**. Solo cambió la forma de escribir la regresión; las predicciones son las mismas.

## 4. Política de educación mínima de 12 años

Para quienes tienen menos de 12 años de educación:

- aumentamos `educ` hasta 12;
- reducimos `exper` en la misma cantidad;
- aplicamos al salario observado el cambio porcentual predicho por el modelo.

In [5]:
b = modelo.params
aumento = (12 - datos['educ']).clip(lower=0)
exper_nueva = datos['exper'] - aumento

cambio_log = (
    b['educ'] * aumento
    - b['exper'] * aumento
    + b['exper2'] * (exper_nueva ** 2 - datos['exper2'])
)
salario_politica = datos['wage'] * np.exp(cambio_log)
efecto_politica = (salario_politica - datos['wage']).mean()

print(f'Personas afectadas: {(aumento > 0).sum()}')
print(f'Salario promedio actual: {datos["wage"].mean():.2f}')
print(f'Salario promedio con la política: {salario_politica.mean():.2f}')
print(f'Efecto promedio: {efecto_politica:.2f}')

Personas afectadas: 88
Salario promedio actual: 957.95
Salario promedio con la política: 966.92
Efecto promedio: 8.98


**Conclusión:** la política afecta a **88 personas**. El salario promedio predicho aumenta de **957,95** a **966,92**; el efecto promedio es **8,98 unidades de salario**, cerca de **0,94 %**.

## 5. Error estándar mediante bootstrap

Repetimos todo el cálculo en 1.000 muestras aleatorias. El error estándar es la desviación estándar de los 1.000 efectos.

In [6]:
def calcular_efecto(muestra):
    muestra = muestra.copy()
    muestra['exper2'] = muestra['exper'] ** 2
    X_b = sm.add_constant(muestra[['educ', 'exper', 'exper2']])
    modelo_b = sm.OLS(muestra['lwage'], X_b).fit()

    aumento_b = (12 - muestra['educ']).clip(lower=0)
    exper_nueva_b = muestra['exper'] - aumento_b
    b_b = modelo_b.params
    cambio_b = (
        b_b['educ'] * aumento_b
        - b_b['exper'] * aumento_b
        + b_b['exper2'] * (exper_nueva_b ** 2 - muestra['exper2'])
    )
    salario_nuevo_b = muestra['wage'] * np.exp(cambio_b)
    return (salario_nuevo_b - muestra['wage']).mean()

np.random.seed(123)
efectos = []
for _ in range(1000):
    muestra = datos.sample(len(datos), replace=True)
    efectos.append(calcular_efecto(muestra))

error_estandar = np.std(efectos, ddof=1)
intervalo = np.percentile(efectos, [2.5, 97.5])

print(f'Error estándar bootstrap: {error_estandar:.2f}')
print(f'Intervalo de confianza 95%: [{intervalo[0]:.2f}, {intervalo[1]:.2f}]')

Error estándar bootstrap: 1.31
Intervalo de confianza 95%: [6.59, 11.57]


**Conclusión:** el error estándar bootstrap es aproximadamente **1,31** unidades de salario. El intervalo de confianza al 95 % es aproximadamente **[6,59; 11,57]**.

> Estos resultados son predicciones del modelo MCO. Una interpretación causal exige que no existan variables omitidas relacionadas simultáneamente con educación y salario.